In [ ]:
# Part A
# Softmax from logits to probabilities

import torch
import torch.nn as nn

In [ ]:
# logits = сырые оценки модели
# это еще не probabilities

logits = torch.tensor([2.0, 1.0, 0.1])

print(logits)

tensor([2.0000, 1.0000, 0.1000])


In [ ]:
# применяем exp к каждому числу
# часть формулы softmax

exp_logits = torch.exp(logits)

print(exp_logits)

tensor([7.3891, 2.7183, 1.1052])


In [ ]:
# сумма всех exp значений
# это знаменатель формулы

denominator = exp_logits.sum()

print(denominator)

tensor(11.2125)


In [ ]:
# делим каждое значение на сумму
# получаем probabilities

manual_probs = exp_logits / denominator

print(manual_probs)

tensor([0.6590, 0.2424, 0.0986])


In [ ]:
# probabilities должны давать сумму 1

print(manual_probs.sum())

tensor(1.0000)


In [ ]:
# сравниваем с softmax из pytorch

torch_probs = torch.softmax(logits, dim=0)

print(torch_probs)

tensor([0.6590, 0.2424, 0.0986])


a2

In [ ]:
# большие logits
# exp(1000) может вызвать overflow

large_logits = torch.tensor([1000.0, 999.0, 998.0])

print(large_logits)

tensor([1000.,  999.,  998.])


In [ ]:
# вычитаем максимальный logit
# так вычисления стабильнее

shifted = large_logits - large_logits.max()

print(shifted)

tensor([ 0., -1., -2.])


In [ ]:
# считаем stable softmax

exp_shifted = torch.exp(shifted)

stable_probs = exp_shifted / exp_shifted.sum()

print(stable_probs)

tensor([0.6652, 0.2447, 0.0900])


In [ ]:
# проверяем сумму probabilities

print(stable_probs.sum())

tensor(1.)


In [ ]:
# сравнение с pytorch softmax

print(torch.softmax(large_logits, dim=0))

tensor([0.6652, 0.2447, 0.0900])


## A2 Answers

### 1. Why do we subtract the maximum logit?

We subtract the maximum logit to avoid very large numbers during exp calculation.  
This makes softmax numerically stable.

---

### 2. Does subtracting the same number from every logit change the final probabilities?

No.  
The final softmax probabilities stay the same.

---

### 3. Which class has the highest probability, and why?

Class 0 has the highest probability because it has the largest logit value.

a3

In [ ]:
# logits для 3 классов

logits = torch.tensor([[2.0, 1.0, 0.1]])

print(logits)

tensor([[2.0000, 1.0000, 0.1000]])


In [ ]:
# правильный класс = 0

true_class = torch.tensor([0])

print(true_class)

tensor([0])


In [ ]:
# функция ошибки

criterion = nn.CrossEntropyLoss()

In [ ]:
# считаем loss

loss = criterion(logits, true_class)

print(loss.item())

0.4170299470424652


In [ ]:
# probabilities для просмотра результата

probs = torch.softmax(logits, dim=1)

print(probs)

tensor([[0.6590, 0.2424, 0.0986]])


In [ ]:
# какой класс выбрала модель

predicted_class = torch.argmax(probs, dim=1)

print(predicted_class.item())

0


## A3 Answers

### 1. Why does CrossEntropyLoss expect logits instead of already-softmaxed probabilities?

CrossEntropyLoss applies softmax internally in a more stable way.  
So we give logits directly.

---

### 2. What happens to the loss when the model assigns higher probability to the true class?

The loss becomes smaller because the prediction is better.

---

### 3. What happens to the loss when the model is confident but wrong?

The loss becomes very large because the model strongly predicted the wrong class.

Part B1 — Load and clean data

In [ ]:
# Part B
# Autoregressive Character RNN

import pandas as pd
import re
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence

In [ ]:
# загружаем csv файл
# в этом файле берем текстовые значения для генерации

df = pd.read_csv("StateGrants.csv", header=None)

df.head()

,0
0,РАХМЕТУЛЛА АСАНӘЛІ
1,Жамашев Ержан Жеңісұлы
2,ЗАМАНБЕК МУХАММЕД ЖЕҢІСҰЛЫ
3,ЖҰМАЖАНОВ МИРАС МАРАТҰЛЫ
4,Тлеулесов Максут Кайратович


In [ ]:
# смотрим названия колонок
# так понимаем, какой столбец использовать

print(df.columns)

Index([0], dtype='int64')


In [ ]:
text_col = 0

raw_names = df[text_col].astype(str).tolist()

print(raw_names[:10])

['РАХМЕТУЛЛА АСАНӘЛІ', 'Жамашев Ержан Жеңісұлы', 'ЗАМАНБЕК МУХАММЕД ЖЕҢІСҰЛЫ', 'ЖҰМАЖАНОВ МИРАС МАРАТҰЛЫ', 'Тлеулесов Максут Кайратович', 'ДҮЙСЕНБЕК НҰРДӘУЛЕТ ЖАҚСЫЛЫҚҰЛЫ', 'ТӨЛЕПБЕРГЕН ҒАЛЫМЖАН АЛТЫНБАЙҰЛЫ', 'САТЫПАЛДИЕВ БАУЫРЖАН НҰРСҰЛТАНҰЛЫ', 'Шинкарова Елена Юрьевна', 'Досмаханбет Ербосын Жанқозыбиұлы']


In [ ]:
# чистим данные:
# 1) lowercase
# 2) убираем лишние пробелы
# 3) оставляем строки нормальной длины

names = []

for name in raw_names:
    name = name.lower()
    name = name.strip()
    name = re.sub(r"\s+", " ", name)

    if len(name) >= 3:
        names.append(name)

print("Number of names:", len(names))
print(names[:10])

Number of names: 2455
['рахметулла асанәлі', 'жамашев ержан жеңісұлы', 'заманбек мухаммед жеңісұлы', 'жұмажанов мирас маратұлы', 'тлеулесов максут кайратович', 'дүйсенбек нұрдәулет жақсылықұлы', 'төлепберген ғалымжан алтынбайұлы', 'сатыпалдиев бауыржан нұрсұлтанұлы', 'шинкарова елена юрьевна', 'досмаханбет ербосын жанқозыбиұлы']


In [ ]:
# собираем все уникальные символы из наших строк

chars = sorted(set("".join(names)))

print("Number of unique chars:", len(chars))
print(chars)

Number of unique chars: 44
[' ', '-', 'а', 'б', 'в', 'г', 'д', 'е', 'ж', 'з', 'и', 'й', 'к', 'л', 'м', 'н', 'о', 'п', 'р', 'с', 'т', 'у', 'ф', 'х', 'ц', 'ч', 'ш', 'щ', 'ъ', 'ы', 'ь', 'э', 'ю', 'я', 'ё', 'і', 'ғ', 'қ', 'ң', 'ү', 'ұ', 'һ', 'ә', 'ө']


In [ ]:
# специальные токены:
# <PAD> нужен для выравнивания длины
# <S> начало строки
# <F> конец строки
# <UNK> неизвестный символ

char2idx = {
    "<PAD>": 0,
    "<S>": 1,
    "<F>": 2,
    "<UNK>": 3
}

In [ ]:
# добавляем обычные символы в vocabulary

for ch in chars:
    if ch not in char2idx:
        char2idx[ch] = len(char2idx)

idx2char = {i: ch for ch, i in char2idx.items()}

vocab_size = len(char2idx)
pad_idx = char2idx["<PAD>"]

print("Vocab size:", vocab_size)
print(char2idx)

Vocab size: 48
{'<PAD>': 0, '<S>': 1, '<F>': 2, '<UNK>': 3, ' ': 4, '-': 5, 'а': 6, 'б': 7, 'в': 8, 'г': 9, 'д': 10, 'е': 11, 'ж': 12, 'з': 13, 'и': 14, 'й': 15, 'к': 16, 'л': 17, 'м': 18, 'н': 19, 'о': 20, 'п': 21, 'р': 22, 'с': 23, 'т': 24, 'у': 25, 'ф': 26, 'х': 27, 'ц': 28, 'ч': 29, 'ш': 30, 'щ': 31, 'ъ': 32, 'ы': 33, 'ь': 34, 'э': 35, 'ю': 36, 'я': 37, 'ё': 38, 'і': 39, 'ғ': 40, 'қ': 41, 'ң': 42, 'ү': 43, 'ұ': 44, 'һ': 45, 'ә': 46, 'ө': 47}


In [ ]:
# функция переводит строку в список индексов

def encode_text(text):
    ids = []

    for ch in text:
        ids.append(char2idx.get(ch, char2idx["<UNK>"]))

    return ids

In [ ]:
# для каждого имени делаем:
# input:  <S> + name
# target: name + <F>

examples = []

for name in names:
    input_ids = [char2idx["<S>"]] + encode_text(name)
    target_ids = encode_text(name) + [char2idx["<F>"]]

    examples.append((torch.tensor(input_ids), torch.tensor(target_ids)))

print("One example:")
print("Input ids:", examples[0][0])
print("Target ids:", examples[0][1])

One example:
Input ids: tensor([ 1, 22,  6, 27, 18, 11, 24, 25, 17, 17,  6,  4,  6, 23,  6, 19, 46, 17,
        39])
Target ids: tensor([22,  6, 27, 18, 11, 24, 25, 17, 17,  6,  4,  6, 23,  6, 19, 46, 17, 39,
         2])


In [ ]:
# покажем этот же пример как символы

x_example, y_example = examples[0]

input_chars = [idx2char[i.item()] for i in x_example]
target_chars = [idx2char[i.item()] for i in y_example]

print("Input chars:", input_chars)
print("Target chars:", target_chars)

Input chars: ['<S>', 'р', 'а', 'х', 'м', 'е', 'т', 'у', 'л', 'л', 'а', ' ', 'а', 'с', 'а', 'н', 'ә', 'л', 'і']
Target chars: ['р', 'а', 'х', 'м', 'е', 'т', 'у', 'л', 'л', 'а', ' ', 'а', 'с', 'а', 'н', 'ә', 'л', 'і', '<F>']


In [ ]:
# Dataset просто хранит наши input и target пары

class CharDataset(Dataset):
    def __init__(self, examples):
        self.examples = examples

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        return self.examples[idx]

In [ ]:
# collate_fn нужен для padding внутри batch
# потому что строки имеют разную длину

def collate_batch(batch):
    x_list = []
    y_list = []

    for x, y in batch:
        x_list.append(x)
        y_list.append(y)

    x_padded = pad_sequence(x_list, batch_first=True, padding_value=pad_idx)
    y_padded = pad_sequence(y_list, batch_first=True, padding_value=pad_idx)

    return x_padded, y_padded

In [ ]:
import random

# делим данные на train и validation

random.shuffle(examples)

split_idx = int(len(examples) * 0.8)

train_examples = examples[:split_idx]
val_examples = examples[split_idx:]

train_dataset = CharDataset(train_examples)
val_dataset = CharDataset(val_examples)

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    collate_fn=collate_batch
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False,
    collate_fn=collate_batch
)

print("Train size:", len(train_dataset))
print("Validation size:", len(val_dataset))

Train size: 1964
Validation size: 491


B2

In [ ]:
import torch
import torch.nn as nn

In [ ]:
# модель для генерации текста по символам

class CharRNN(nn.Module):

    def __init__(self, vocab_size, embed_dim, hidden_size, pad_idx):
        super().__init__()

        # embedding превращает id символов в vectors
        self.embedding = nn.Embedding(
            vocab_size,
            embed_dim,
            padding_idx=pad_idx
        )

        # GRU читает последовательность символов
        self.rnn = nn.GRU(
            embed_dim,
            hidden_size,
            batch_first=True
        )

        # linear layer предсказывает следующий символ
        self.fc = nn.Linear(hidden_size, vocab_size)

    def forward(self, x, hidden=None):

        # shape:
        # (batch_size, seq_len)
        embedded = self.embedding(x)

        # output:
        # (batch_size, seq_len, hidden_size)
        output, hidden = self.rnn(embedded, hidden)

        # logits:
        # (batch_size, seq_len, vocab_size)
        logits = self.fc(output)

        return logits, hidden

In [ ]:
# создаем модель

embed_dim = 32
hidden_size = 64

model = CharRNN(
    vocab_size=vocab_size,
    embed_dim=embed_dim,
    hidden_size=hidden_size,
    pad_idx=pad_idx
)

print(model)

CharRNN(
  (embedding): Embedding(48, 32, padding_idx=0)
  (rnn): GRU(32, 64, batch_first=True)
  (fc): Linear(in_features=64, out_features=48, bias=True)
)


In [ ]:
# берем один batch

x_batch, y_batch = next(iter(train_loader))

print("x_batch shape:", x_batch.shape)

x_batch shape: torch.Size([32, 34])


In [ ]:
# пропускаем batch через модель

logits, hidden = model(x_batch)

In [ ]:
# logits:
# (batch_size, seq_len, vocab_size)

print("logits shape:", logits.shape)

logits shape: torch.Size([32, 34, 48])


In [ ]:
# hidden:
# (num_layers, batch_size, hidden_size)

print("hidden shape:", hidden.shape)

hidden shape: torch.Size([1, 32, 64])


b3

In [ ]:
# loss function
# ignore_index=pad_idx значит, что модель не будет считать ошибку на <PAD>

criterion = nn.CrossEntropyLoss(ignore_index=pad_idx)

In [ ]:
# optimizer обновляет веса модели во время training

optimizer = torch.optim.Adam(model.parameters(), lr=0.003)

In [ ]:
# выбираем device
# если есть GPU, используем cuda

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = model.to(device)

print(device)

cpu


In [ ]:
# обучаем модель несколько эпох

num_epochs = 20

for epoch in range(num_epochs):

    model.train()
    total_loss = 0

    for x_batch, y_batch in train_loader:

        # переносим batch на device
        x_batch = x_batch.to(device)
        y_batch = y_batch.to(device)

        # очищаем старые gradients
        optimizer.zero_grad()

        # forward pass
        logits, _ = model(x_batch)

        # logits shape: (batch, seq_len, vocab_size)
        # y_batch shape: (batch, seq_len)
        #
        # CrossEntropyLoss хочет:
        # logits: (N, vocab_size)
        # target: (N)
        #
        # поэтому делаем reshape

        loss = criterion(
            logits.reshape(-1, vocab_size),
            y_batch.reshape(-1)
        )

        # backward pass
        loss.backward()

        # обновляем веса модели
        optimizer.step()

        total_loss += loss.item()

    avg_train_loss = total_loss / len(train_loader)

    print(f"Epoch {epoch+1}/{num_epochs}, Train Loss: {avg_train_loss:.4f}")

Epoch 1/20, Train Loss: 2.8309
Epoch 2/20, Train Loss: 2.1098
Epoch 3/20, Train Loss: 1.9134
Epoch 4/20, Train Loss: 1.8007
Epoch 5/20, Train Loss: 1.7082
Epoch 6/20, Train Loss: 1.6347
Epoch 7/20, Train Loss: 1.5820
Epoch 8/20, Train Loss: 1.5386
Epoch 9/20, Train Loss: 1.5042
Epoch 10/20, Train Loss: 1.4746
Epoch 11/20, Train Loss: 1.4510
Epoch 12/20, Train Loss: 1.4272
Epoch 13/20, Train Loss: 1.4064
Epoch 14/20, Train Loss: 1.3889
Epoch 15/20, Train Loss: 1.3717
Epoch 16/20, Train Loss: 1.3577
Epoch 17/20, Train Loss: 1.3458
Epoch 18/20, Train Loss: 1.3307
Epoch 19/20, Train Loss: 1.3178
Epoch 20/20, Train Loss: 1.3076


In [ ]:
# считаем validation next-character accuracy
# padding не учитываем

model.eval()

correct = 0
total = 0

with torch.no_grad():

    for x_batch, y_batch in val_loader:

        x_batch = x_batch.to(device)
        y_batch = y_batch.to(device)

        logits, _ = model(x_batch)

        preds = torch.argmax(logits, dim=-1)

        # mask показывает, где НЕ padding
        mask = y_batch != pad_idx

        correct += ((preds == y_batch) & mask).sum().item()
        total += mask.sum().item()

val_accuracy = correct / total

print("Validation next-character accuracy:", val_accuracy)

Validation next-character accuracy: 0.5687196485007323


In [ ]:
# берем один validation batch

model.eval()

x_batch, y_batch = next(iter(val_loader))

x_batch = x_batch.to(device)
y_batch = y_batch.to(device)

with torch.no_grad():
    logits, _ = model(x_batch)
    preds = torch.argmax(logits, dim=-1)

In [ ]:
# берем первый пример из batch

x_one = x_batch[0].cpu()
y_one = y_batch[0].cpu()
pred_one = preds[0].cpu()

In [ ]:
# переводим ids обратно в символы

input_chars = []
target_chars = []
pred_chars = []

for i in range(len(x_one)):

    if x_one[i].item() != pad_idx:
        input_chars.append(idx2char[x_one[i].item()])

    if y_one[i].item() != pad_idx:
        target_chars.append(idx2char[y_one[i].item()])

    if y_one[i].item() != pad_idx:
        pred_chars.append(idx2char[pred_one[i].item()])

In [ ]:
print("Input characters:")
print(input_chars)

print("\nTarget characters:")
print(target_chars)

print("\nPredicted characters:")
print(pred_chars)

Input characters:
['<S>', 'қ', 'ы', 'р', 'ғ', 'ы', 'з', 'б', 'а', 'й', ' ', 'ә', 'л', 'н', 'ұ', 'р', ' ', 'б', 'а', 'у', 'ы', 'р', 'ж', 'а', 'н', 'ұ', 'л', 'ы']

Target characters:
['қ', 'ы', 'р', 'ғ', 'ы', 'з', 'б', 'а', 'й', ' ', 'ә', 'л', 'н', 'ұ', 'р', ' ', 'б', 'а', 'у', 'ы', 'р', 'ж', 'а', 'н', 'ұ', 'л', 'ы', '<F>']

Predicted characters:
['а', 'а', 'д', 'ж', 'ы', 'с', 'б', 'а', 'е', ' ', 'а', 'л', 'і', 'а', 'р', ' ', 'м', 'а', 'у', 'ы', 'р', 'ж', 'а', 'н', 'ұ', 'л', 'ы', '<F>']


## B3. Training Explanation

In this part, I trained the character-level RNN to predict the next character.

The input sequence was `<S> + text`, and the target sequence was `text + <F>`.  
So at each position, the model learned to predict the next character.

I used CrossEntropyLoss because this is a classification problem over the character vocabulary.  
I used `ignore_index=pad_idx` because `<PAD>` is only used for padding and should not affect the loss.

The model output shape was:

`(batch_size, max_seq_len, vocab_size)`

Before calculating the loss, I reshaped logits and targets because CrossEntropyLoss expects:

`(N, vocab_size)` for logits and `(N)` for targets.

After training, I checked validation next-character accuracy and printed one example with input, target, and predicted characters.

Part C — Generate Text Autoregressively

In [ ]:
# функция для генерации текста
# модель начинает с <S> и дальше сама выбирает символы

def generate(start_text="", max_len=30, temperature=1.0, greedy=False):

    model.eval()

    # hidden нужен, чтобы GRU помнила предыдущие символы
    hidden = None

    # начинаем с токена старта
    current_id = torch.tensor([[char2idx["<S>"]]], device=device)

    generated = ""

    with torch.no_grad():

        # сначала прогоняем prefix, если он есть
        for ch in start_text.lower():

            # берем id символа, если нет такого символа, берем <UNK>
            next_id = char2idx.get(ch, char2idx["<UNK>"])

            # подаем текущий символ в модель
            logits, hidden = model(current_id, hidden)

            # теперь следующим input будет символ из prefix
            current_id = torch.tensor([[next_id]], device=device)

            generated += ch

        # теперь модель сама генерирует продолжение
        for _ in range(max_len):

            logits, hidden = model(current_id, hidden)

            # берем logits только последнего шага
            last_logits = logits[:, -1, :]

            if greedy:
                # greedy = берем самый вероятный символ
                next_id = torch.argmax(last_logits, dim=-1).item()

            else:
                # temperature управляет случайностью
                probs = torch.softmax(last_logits / temperature, dim=-1)

                # выбираем символ случайно по probabilities
                next_id = torch.multinomial(probs, num_samples=1).item()

            next_char = idx2char[next_id]

            # если модель сказала <F>, заканчиваем генерацию
            if next_char == "<F>":
                break

            # пропускаем специальные токены, чтобы они не попали в текст
            if next_char not in ["<PAD>", "<S>", "<UNK>"]:
                generated += next_char

            # новый символ становится следующим input
            current_id = torch.tensor([[next_id]], device=device)

    return generated

c2


In [ ]:
# greedy decoding
# модель всегда берет самый вероятный символ

prefixes = ["а", "ай", "нур"]

for prefix in prefixes:
    print("Prefix:", prefix)

    for i in range(5):
        text = generate(start_text=prefix, max_len=30, greedy=True)
        print(text)

    print()

Prefix: а
абдуасын арайлым бауыржанұлы
абдуасын арайлым бауыржанұлы
абдуасын арайлым бауыржанұлы
абдуасын арайлым бауыржанұлы
абдуасын арайлым бауыржанұлы

Prefix: ай
айткен алихан александрович
айткен алихан александрович
айткен алихан александрович
айткен алихан александрович
айткен алихан александрович

Prefix: нур
нургалиев алихан александрович
нургалиев алихан александрович
нургалиев алихан александрович
нургалиев алихан александрович
нургалиев алихан александрович



c3

In [ ]:
# temperature 0.5
# меньше случайности, результаты обычно более безопасные

for prefix in prefixes:
    print("Prefix:", prefix)

    for i in range(5):
        text = generate(start_text=prefix, max_len=30, temperature=0.5, greedy=False)
        print(text)

    print()

Prefix: а
абдулас ержан алекселіханұлы
абитов алима сабитқызы
алман бауыржан данияровна
абдуали арайлым русланұлы
абдошов ақылтан байболатұлы

Prefix: ай
айткен аружан бауыржанұлы
айболат қайрат маратұлы
айдарханов дархин мадила ермекқы
айпар бауыржан нұрланұлы
айпар мадияр ержанұлы

Prefix: нур
нуразин ерке ерболатұлы
нуржан бақтыржан маратұлы
нураз алихан нұрланұлы
нургалиев алихан ержанұлы
нурали александр русланович



c4

In [ ]:
# temperature 1.0
# обычный баланс между стабильностью и случайностью

for prefix in prefixes:
    print("Prefix:", prefix)

    for i in range(5):
        text = generate(start_text=prefix, max_len=30, temperature=1.0, greedy=False)
        print(text)

    print()

Prefix: а
адилханов андаржан жазжанқызы
алман арман ержанович
абдаулетбекова айдәулетхан нұрл
абдыутов серік сабатқызы
абдутеменова динара ахметовна

Prefix: ай
аймұлтая балтышбаева дануллинұлы
айтбай ниязля
айдияр емаргельды жолғамбетұлы
айболатов мейрам талгатовна
айтұрак ерасыл қайратұлы

Prefix: нур
нурболатқан тасгарик мекмахатовна
нургалиев ахат абдуғалиевна
нураза мархамұлы ба маилбетұлы
нуртарісбаева дилина ризалыұлы
нуртөп намир



c5

In [ ]:
# temperature 1.5
# больше случайности, иногда выходят странные варианты

for prefix in prefixes:
    print("Prefix:", prefix)

    for i in range(5):
        text = generate(start_text=prefix, max_len=30, temperature=1.5, greedy=False)
        print(text)

    print()

Prefix: а
алдыгаев нұрмінгулда іглов ұлжа
азджерді нулгам кегендолұлы
ахбыққо
аемалова спиткім чидильвич
аминдақозаевна арген

Prefix: ай
айсанбет сулекхан латөвұфа
аймбетотшолов масар
айтбаила уванхариқыз тебитбібекұ
айеленғаля арүұлетовиа гигзалықы
айдұлтанбаев арастсхан досбакқыз

Prefix: нур
нурдасова данитан евназбаевна
нурқапколова жасығара омагұлжанов
нурленкуза кенхботов алдос
нурубар нұрлығазы канатуайлы
нурчаже деурза қуастыбчетболбеула



## Part C Answers

### 1. What changes when temperature increases?

When temperature increases, the generation becomes more random.  
The model starts choosing not only the most likely characters, but also less likely characters.  
Because of this, the generated text becomes more diverse, but it can also contain more mistakes.

---

### 2. Why does greedy decoding often repeat safe or common characters?

Greedy decoding always chooses the character with the highest probability.  
Because of this, the model often chooses common and safe characters.  
It does not explore other possible characters, so the result can become repetitive.

---

### 3. What kinds of mistakes does the model make when it feeds its own predictions back into itself?

The model can make spelling mistakes or generate unrealistic names.  
If it predicts one wrong character, this wrong character becomes the next input.  
Because of this, the error can continue and affect the rest of the generated text.

---

### 4. How is generation different from training with the correct previous characters?

During training, the model receives the correct previous characters from the dataset.  
During generation, the model uses its own predicted characters as the next input.  
So generation is harder because the model has to continue from its own outputs.

Д

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
import random

In [ ]:
ner_data = [
    (["Apple", "opened", "an", "office", "in", "Almaty"],
     ["B-ORG", "O", "O", "O", "O", "B-LOC"]),

    (["Narxoz", "University", "is", "in", "Almaty"],
     ["B-ORG", "I-ORG", "O", "O", "B-LOC"]),

    (["Aruzhan", "studies", "at", "Narxoz", "University"],
     ["B-PER", "O", "O", "B-ORG", "I-ORG"]),

    (["Google", "has", "an", "office", "in", "London"],
     ["B-ORG", "O", "O", "O", "O", "B-LOC"]),

    (["Ali", "works", "at", "Microsoft", "in", "Astana"],
     ["B-PER", "O", "O", "B-ORG", "O", "B-LOC"]),

    (["Kazakhstan", "is", "a", "country"],
     ["B-LOC", "O", "O", "O"]),

    (["Dana", "visited", "Paris"],
     ["B-PER", "O", "B-LOC"]),

    (["Samsung", "released", "a", "new", "phone"],
     ["B-ORG", "O", "O", "O", "O"]),

    (["Aigerim", "lives", "in", "Shymkent"],
     ["B-PER", "O", "O", "B-LOC"]),

    (["KBTU", "is", "located", "in", "Almaty"],
     ["B-ORG", "O", "O", "O", "B-LOC"]),

    (["New", "York", "is", "big"],
     ["B-LOC", "I-LOC", "O", "O"]),

    (["John", "Smith", "works", "at", "Google"],
     ["B-PER", "I-PER", "O", "O", "B-ORG"]),
]

In [ ]:
PAD_TOKEN = "<PAD>"
UNK_TOKEN = "<UNK>"
PAD_TAG = "<PAD_TAG>"

In [ ]:
all_words = set()
all_tags = set()

for tokens, tags in ner_data:
    for w in tokens:
        all_words.add(w.lower())
    for t in tags:
        all_tags.add(t)

all_words = sorted(all_words)
all_tags = sorted(all_tags)

all_tags.append(PAD_TAG)

In [ ]:
word2idx = {
    PAD_TOKEN: 0,
    UNK_TOKEN: 1
}

for w in all_words:
    word2idx[w] = len(word2idx)

tag2idx = {
    PAD_TAG: 0
}

for t in all_tags:
    if t != PAD_TAG:
        tag2idx[t] = len(tag2idx)

idx2word = {i: w for w, i in word2idx.items()}
idx2tag = {i: t for t, i in tag2idx.items()}

In [ ]:
ner_examples = []

for tokens, tags in ner_data:

    assert len(tokens) == len(tags)

    word_ids = [word2idx.get(w.lower(), word2idx[UNK_TOKEN]) for w in tokens]
    tag_ids = [tag2idx[t] for t in tags]

    ner_examples.append((
        torch.tensor(word_ids, dtype=torch.long),
        torch.tensor(tag_ids, dtype=torch.long)
    ))

In [ ]:
pad_word_id = word2idx[PAD_TOKEN]
pad_tag_id = tag2idx[PAD_TAG]

def ner_collate_batch(batch):

    x_list = []
    y_list = []

    for x, y in batch:
        x_list.append(x)
        y_list.append(y)

    x_padded = pad_sequence(x_list, batch_first=True, padding_value=pad_word_id)
    y_padded = pad_sequence(y_list, batch_first=True, padding_value=pad_tag_id)

    return x_padded, y_padded


class NERDataset(Dataset):
    def __init__(self, examples):
        self.examples = examples

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        return self.examples[idx]

In [ ]:
random.shuffle(ner_examples)

split = int(0.8 * len(ner_examples))

train_data = ner_examples[:split]
val_data = ner_examples[split:]

train_loader = DataLoader(train_data, batch_size=4, shuffle=True, collate_fn=ner_collate_batch)
val_loader = DataLoader(val_data, batch_size=4, shuffle=False, collate_fn=ner_collate_batch)

In [ ]:
class BiRNNNER(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_size, num_tags, pad_idx):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)

        self.rnn = nn.LSTM(
            embed_dim,
            hidden_size,
            batch_first=True,
            bidirectional=True
        )

        self.fc = nn.Linear(hidden_size * 2, num_tags)

    def forward(self, x):
        x = self.embedding(x)
        out, _ = self.rnn(x)
        logits = self.fc(out)
        return logits

In [ ]:
model = BiRNNNER(
    vocab_size=len(word2idx),
    embed_dim=32,
    hidden_size=64,
    num_tags=len(tag2idx),
    pad_idx=word2idx[PAD_TOKEN]
)

In [ ]:
criterion = nn.CrossEntropyLoss(ignore_index=pad_tag_id)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

In [ ]:
for epoch in range(10):

    model.train()
    total_loss = 0

    for x_batch, y_batch in train_loader:

        logits = model(x_batch)

        loss = criterion(
            logits.reshape(-1, len(tag2idx)),
            y_batch.reshape(-1)
        )

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch}: loss = {total_loss:.4f}")

Epoch 0: loss = 6.0925
Epoch 1: loss = 3.8302
Epoch 2: loss = 2.5694
Epoch 3: loss = 1.4807
Epoch 4: loss = 0.8928
Epoch 5: loss = 0.9271
Epoch 6: loss = 0.3089
Epoch 7: loss = 0.1805
Epoch 8: loss = 0.1197
Epoch 9: loss = 0.0797


In [ ]:
model.eval()

correct = 0
total = 0

with torch.no_grad():
    for x_batch, y_batch in val_loader:

        logits = model(x_batch)
        preds = torch.argmax(logits, dim=-1)

        mask = (y_batch != pad_tag_id)

        correct += ((preds == y_batch) * mask).sum().item()
        total += mask.sum().item()

print("Validation accuracy:", correct / total)

Validation accuracy: 0.7142857142857143


In [ ]:
def show_examples(model, loader, num=5):
    model.eval()
    count = 0

    with torch.no_grad():
        for x_batch, y_batch in loader:

            logits = model(x_batch)
            preds = torch.argmax(logits, dim=-1)

            for i in range(x_batch.size(0)):

                print("\nSentence:")
                for j in range(x_batch.size(1)):

                    if y_batch[i][j] == pad_tag_id:
                        continue

                    word = idx2word[x_batch[i][j].item()]
                    true_tag = idx2tag[y_batch[i][j].item()]
                    pred_tag = idx2tag[preds[i][j].item()]

                    print(f"{word:10} {true_tag:8} {pred_tag:8}")

                count += 1
                if count >= num:
                    return

show_examples(model, val_loader)


Sentence:
ali        B-PER    B-PER   
works      O        O       
at         O        O       
microsoft  B-ORG    B-ORG   
in         O        B-ORG   
astana     B-LOC    I-ORG   

Sentence:
aigerim    B-PER    B-PER   
lives      O        O       
in         O        O       
shymkent   B-LOC    B-LOC   

Sentence:
new        B-LOC    O       
york       I-LOC    O       
is         O        O       
big        O        O       


E
| Task                 | Input               | Target             | Output shape                      | Uses softmax over | Decoding method   |
| -------------------- | ------------------- | ------------------ | --------------------------------- | ----------------- | ----------------- |
| Character generation | previous characters | next character     | (batch_size, seq_len, vocab_size) | vocabulary        | sampling / argmax |
| NER                  | sentence tokens     | tag for each token | (batch_size, seq_len, num_tags)   | tag set           | argmax            |

1. **Why is character generation autoregressive?**

Character generation is autoregressive because the model predicts the next character using previous characters.
Each new character depends on the ones before.

2. **Why is NER not autoregressive?**

NER is not autoregressive because we predict all tags at the same time.
Each token gets a tag directly, not step by step.

3. **Why is the final hidden state not enough for NER?**

The final hidden state gives only one vector.
But NER needs one tag for each token.
So we need outputs for every time step.

4. **Why do both tasks use cross-entropy loss?**

Both tasks are classification problems.

Character generation → choose one character
NER → choose one tag

Cross-entropy measures how correct the prediction is.

5. **What does softmax help us interpret?**

Softmax converts logits into probabilities.
It shows how confident the model is for each class.